# Review 2 Kaggle GPU smoke run

This notebook checks the existing data, Transformer, and confidence novelty interfaces on a small subset of the **combined flow collection**. The Parquet file has no verified per-row source provenance. This is a functional smoke run, not final training, continual learning, poisoning, or an unknown-attack performance study. Synthetic data below is used only for a functional check.

In Kaggle, create a new notebook, use **File → Import Notebook** to upload this `.ipynb`, then set **Internet: On** and **Accelerator: GPU** in notebook settings. Upload `cic-collection.parquet` as a private Kaggle Dataset and attach it with **Add Input**. Kaggle mounts it under `/kaggle/input/`; outputs will be written under `/kaggle/working/`. Run the cells in order. See [Kaggle's notebook documentation](https://www.kaggle.com/docs/notebooks) for the current editor controls.


## 1. Check out the verified source revision

The model, data pipeline, and novelty package were verified together on the published `feature/unified-novelty` branch at the full SHA below. A detached checkout makes this run repeatable even if the branch receives later commits. Internet is needed for this step.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/DevChi1l/poison-resilient-continual-nids.git'
PUBLISHED_BRANCH = 'feature/unified-novelty'
SOURCE_SHA = '22e797660307bfed62b739d61b5e4c05bcb0a187'
REPO_DIR = Path('/kaggle/working/poison-resilient-continual-nids')
if REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} already exists. Use a fresh Kaggle session or inspect it before rerunning this cell.')
subprocess.run(['git', 'clone', '--branch', PUBLISHED_BRANCH, '--single-branch', REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO_DIR, check=True)
checked_out_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert checked_out_sha == SOURCE_SHA
sys.path.insert(0, str(REPO_DIR))
print('Checked out source SHA:', checked_out_sha)


## 2. Inspect Kaggle packages and compute

Kaggle usually provides a CUDA-enabled PyTorch build. This cell records versions and checks imports without replacing that build or blindly installing the repository's local dependency pins. Resolve a reported incompatibility in the Kaggle environment before proceeding. GPU is required by default; change `ALLOW_CPU_FALLBACK` only when intentionally testing on CPU.


In [ ]:
from importlib.metadata import version
import platform
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch

ALLOW_CPU_FALLBACK = False  # Set True only for an intentional CPU run.
cuda_available = bool(torch.cuda.is_available())
gpu_name = torch.cuda.get_device_name(0) if cuda_available else None
if not cuda_available and not ALLOW_CPU_FALLBACK:
    raise RuntimeError('CUDA GPU is unavailable. Enable a Kaggle GPU, or explicitly set ALLOW_CPU_FALLBACK = True.')
device = 'cuda' if cuda_available else 'cpu'
package_versions = {name: version(name) for name in ('numpy', 'pandas', 'pyarrow', 'torch')}
environment = {
    'python': platform.python_version(),
    'packages': package_versions,
    'torch_cuda_build': torch.version.cuda,
    'cuda_available': cuda_available,
    'gpu_name': gpu_name,
    'device_used': device,
}
# Minimal cross-package compatibility checks; these do not install or alter packages.
assert np.asarray([1.0], dtype=np.float32).dtype == np.float32
assert pd.Series([1, 2]).sum() == 3
assert pa.array([1, 2]).to_pylist() == [1, 2]
assert torch.tensor([1.0]).numel() == 1
print('Environment:', environment)
print('Core package compatibility checks: passed')


## 3. Verify model fit and prediction on synthetic inputs

This tiny seeded fit checks that the checked-out classifier trains and predicts in the selected Kaggle environment before reading the large Parquet input. These synthetic values are **functional verification only** and are not NIDS evidence.


In [ ]:
from time import perf_counter
from src.models import ModelConfig, TabularTransformerClassifier
from src.data import prepare_sampled_dataset, sample_class_balanced_indices, stratified_split_indices
from src.novelty import detect_unknown

synthetic_rng = np.random.default_rng(42)
synthetic_X = synthetic_rng.normal(size=(16, 4)).astype(np.float32)
synthetic_y = np.asarray([10, 20] * 8, dtype=np.int64)
synthetic_config = ModelConfig(num_features=4, num_classes=2, hidden_dim=16, num_heads=2, num_layers=1, mlp_dim=32, batch_size=8, epochs=1, device=device, mixed_precision=False, seed=42, early_stopping_patience=None)
synthetic_model = TabularTransformerClassifier(synthetic_config, class_ids=[10, 20])
synthetic_start = perf_counter()
synthetic_history = synthetic_model.fit(synthetic_X[:12], synthetic_y[:12], synthetic_X[12:], synthetic_y[12:])
synthetic_predictions = synthetic_model.predict(synthetic_X[12:])
synthetic_probabilities = synthetic_model.predict_proba(synthetic_X[12:])
synthetic_seconds = perf_counter() - synthetic_start
assert synthetic_history.epochs_completed == 1
assert np.isfinite(synthetic_history.train_loss + synthetic_history.validation_loss).all()
assert synthetic_predictions.shape == (4,) and synthetic_probabilities.shape == (4, 2)
assert np.isfinite(synthetic_probabilities).all()
assert np.allclose(synthetic_probabilities.sum(axis=1), 1.0, atol=1e-5)
assert np.array_equal(synthetic_predictions, np.asarray(synthetic_model.class_ids)[synthetic_probabilities.argmax(axis=1)])
print('Synthetic functional check: passed on', device, 'in', round(synthetic_seconds, 3), 'seconds')


## 4. Locate the attached combined flow collection

Attach your private Kaggle Dataset containing `cic-collection.parquet`. Leave `PARQUET_PATH = None` for automatic discovery. If several files have that name, set the full mounted path explicitly. The cell stops with a clear error for zero or multiple candidates. The file's source-dataset provenance remains unverified.


In [ ]:
PARQUET_PATH = None  # Example: '/kaggle/input/my-private-dataset/cic-collection.parquet'
candidates = sorted(Path('/kaggle/input').rglob('cic-collection.parquet'))
print('Matching attached files:', [str(path) for path in candidates])
if PARQUET_PATH is None:
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one attached cic-collection.parquet; found {len(candidates)}. Attach the file or set PARQUET_PATH to its full /kaggle/input path.')
    parquet_path = candidates[0]
else:
    parquet_path = Path(PARQUET_PATH)
    if not parquet_path.is_file():
        raise FileNotFoundError(f'PARQUET_PATH does not point to a file: {parquet_path}')
    if not parquet_path.is_relative_to(Path('/kaggle/input')):
        raise ValueError('PARQUET_PATH must be an attached file under /kaggle/input')
print('Selected combined flow collection:', parquet_path, 'bytes:', parquet_path.stat().st_size)


## 5. Select 64 rows per broad class and preprocess

The existing seeded reservoir sampler keeps only 64 original row indices per broad class in memory, but it still **scans the full Parquet file** to give each eligible row a selection chance. The existing splitter then makes stratified train/validation/test positions. `prepare_sampled_dataset` scans to materialize selected rows and fits imputation/scaling on training rows only. We check the resulting 54-feature matrices and stable class mapping.


In [ ]:
SAMPLE_SEED = 42
SPLIT_SEED = 42
PER_CLASS_LIMIT = 64
stage_start = perf_counter()
sample = sample_class_balanced_indices(parquet_path, per_class_limit=PER_CLASS_LIMIT, seed=SAMPLE_SEED)
sampling_seconds = perf_counter() - stage_start
stage_start = perf_counter()
split = stratified_split_indices(sample.labels, seed=SPLIT_SEED)
splitting_seconds = perf_counter() - stage_start
stage_start = perf_counter()
dataset = prepare_sampled_dataset(parquet_path, sample, split)
preprocessing_seconds = perf_counter() - stage_start
partitions = {'train': dataset.train, 'validation': dataset.validation, 'test': dataset.test}
for name, partition in partitions.items():
    assert partition.features.dtype == np.float32 and partition.labels.dtype == np.int64
    assert partition.features.shape[1] == 54 and np.isfinite(partition.features).all()
    print(name, 'features:', partition.features.shape, 'labels:', partition.labels.shape, 'finite:', bool(np.isfinite(partition.features).all()))
assert len(dataset.class_names) == 8
print('Broad class ID mapping:', dataset.class_names)
print('Original row counts:', sample.available_counts)
print('Stage seconds:', {'sampling': sampling_seconds, 'splitting': splitting_seconds, 'preprocessing': preprocessing_seconds})


## 6. Train the existing tiny Transformer for one epoch

This uses the repository model API with 54 input features and eight broad classes. GPU is selected unless the earlier explicit CPU fallback was enabled. One epoch and a small network verify the real-data path; their results are not a final model comparison.


In [ ]:
model_config = ModelConfig(num_features=len(dataset.feature_names), num_classes=len(dataset.class_names), hidden_dim=16, num_heads=2, num_layers=1, mlp_dim=32, batch_size=32, epochs=1, device=device, mixed_precision=False, seed=42, early_stopping_patience=None)
model = TabularTransformerClassifier(model_config, class_ids=tuple(sorted(dataset.class_names)))
if device == 'cuda':
    torch.cuda.synchronize()
stage_start = perf_counter()
history = model.fit(dataset.train.features, dataset.train.labels, dataset.validation.features, dataset.validation.labels)
if device == 'cuda':
    torch.cuda.synchronize()
training_seconds = perf_counter() - stage_start
history_dict = history.to_dict()
assert history.epochs_completed == 1
assert np.isfinite(history.train_loss + history.validation_loss + history.validation_accuracy).all()
print('Model device:', model.device)
print('Training history:', history_dict)
print('Training seconds:', training_seconds)


## 7. Check predictions and compute observed smoke metrics

The model's probability column positions map through `model.class_ids` to external IDs. We verify that `predict` agrees with that mapping, then compute test classification metrics directly from the observed labels and predictions. `detect_unknown` flags low-confidence rows; because all eight broad classes were included in training, its flag count is **not** an unknown-attack detection score. The threshold is editable and has not been calibrated.


In [ ]:
NOVELTY_THRESHOLD = 0.50
stage_start = perf_counter()
test_predictions = model.predict(dataset.test.features)
test_probabilities = model.predict_proba(dataset.test.features)
evaluation_seconds = perf_counter() - stage_start
y_true = dataset.test.labels
class_ids = np.asarray(model.class_ids, dtype=np.int64)
assert test_predictions.shape == y_true.shape
assert test_probabilities.shape == (len(y_true), len(class_ids))
assert np.isfinite(test_probabilities).all()
assert np.allclose(test_probabilities.sum(axis=1), 1.0, atol=1e-5)
assert np.array_equal(test_predictions, class_ids[test_probabilities.argmax(axis=1)])
unknown_mask = detect_unknown(test_probabilities, NOVELTY_THRESHOLD)
assert unknown_mask.shape == y_true.shape
confusion = np.asarray([[np.sum((y_true == truth) & (test_predictions == predicted)) for predicted in class_ids] for truth in class_ids], dtype=np.int64)
per_class = {}
for index, class_id in enumerate(class_ids):
    tp = int(confusion[index, index])
    precision = tp / int(confusion[:, index].sum()) if confusion[:, index].sum() else 0.0
    recall = tp / int(confusion[index, :].sum()) if confusion[index, :].sum() else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    per_class[str(int(class_id))] = {'name': dataset.class_names[int(class_id)], 'precision': precision, 'recall': recall, 'f1': f1, 'support': int(confusion[index, :].sum())}
metrics = {'accuracy': float(np.trace(confusion) / len(y_true)), 'macro_f1': float(np.mean([row['f1'] for row in per_class.values()])), 'class_ids': class_ids.tolist(), 'confusion_matrix': confusion.tolist(), 'per_class': per_class}
novelty_counts = {'flagged_unknown': int(unknown_mask.sum()), 'not_flagged': int((~unknown_mask).sum())}
print('Prediction shape:', test_predictions.shape, 'probability shape:', test_probabilities.shape)
print('class_ids in probability-column order:', model.class_ids)
print('Observed test classification metrics:', metrics)
print('Confidence flags only:', novelty_counts, 'threshold:', NOVELTY_THRESHOLD)


## 8. Save the observed run and its inputs

A timestamped folder under `/kaggle/working` keeps the checkpoint, the original selected/split row indices, fitted preprocessing state, metrics, history, and an auditable manifest together. The SHA-256 checksum identifies the attached input file; computing it reads the file once more. Download the output folder from Kaggle after the run. Do not commit generated artifacts or the dataset to Git.


In [ ]:
from dataclasses import asdict
from datetime import datetime, timezone
import hashlib
import json

OUTPUT_DIR = Path('/kaggle/working') / ('review2_smoke_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
checksum_start = perf_counter()
digest = hashlib.sha256()
with parquet_path.open('rb') as source_file:
    for chunk in iter(lambda: source_file.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
input_sha256 = digest.hexdigest()
checksum_seconds = perf_counter() - checksum_start
save_start = perf_counter()
checkpoint_path = OUTPUT_DIR / 'transformer.pt'
model.save(checkpoint_path)
np.savez_compressed(OUTPUT_DIR / 'row_indices.npz', selected_original=sample.row_indices, train_positions=split.train, validation_positions=split.validation, test_positions=split.test, train_original=dataset.train.row_indices, validation_original=dataset.validation.row_indices, test_original=dataset.test.row_indices)
preprocessing_state = {'feature_names': list(dataset.preprocessor.feature_names), 'imputation_values': dataset.preprocessor.imputation_values.tolist(), 'means': dataset.preprocessor.means.tolist(), 'scales': dataset.preprocessor.scales.tolist(), 'class_names': {str(key): value for key, value in dataset.class_names.items()}}
(OUTPUT_DIR / 'preprocessing.json').write_text(json.dumps(preprocessing_state, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'training_history.json').write_text(json.dumps(history_dict, indent=2), encoding='utf-8')
save_seconds = perf_counter() - save_start
manifest = {
    'purpose': 'Kaggle Review-2 functional smoke; all broad classes are known to training',
    'source_sha': checked_out_sha, 'source_branch': PUBLISHED_BRANCH,
    'environment': environment,
    'input': {'path': str(parquet_path), 'bytes': parquet_path.stat().st_size, 'sha256': input_sha256, 'description': 'combined flow collection; per-row source provenance unverified'},
    'sampling': {'per_class_limit': PER_CLASS_LIMIT, 'seed': SAMPLE_SEED, 'class_order': list(sample.class_order), 'available_counts': sample.available_counts},
    'split': {'seed': SPLIT_SEED, 'fractions': {'train': 0.70, 'validation': 0.15, 'test': 0.15}, 'shapes': {name: list(partition.features.shape) for name, partition in partitions.items()}},
    'model_config': asdict(model.config), 'class_ids': list(model.class_ids),
    'novelty_threshold': NOVELTY_THRESHOLD, 'novelty_flag_counts_only': novelty_counts,
    'metrics': metrics, 'training_history': history_dict,
    'timings_seconds': {'synthetic_check': synthetic_seconds, 'sampling': sampling_seconds, 'splitting': splitting_seconds, 'preprocessing': preprocessing_seconds, 'training': training_seconds, 'evaluation': evaluation_seconds, 'input_sha256': checksum_seconds, 'saving': save_seconds},
    'artifacts': {'checkpoint': 'transformer.pt', 'row_indices': 'row_indices.npz', 'preprocessing': 'preprocessing.json', 'metrics': 'metrics.json', 'training_history': 'training_history.json'},
    'checkpoint_reload_verified': False,
}
manifest_path = OUTPUT_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved smoke artifacts to:', OUTPUT_DIR)
print('Input SHA-256:', input_sha256)


## 9. Verify checkpoint reload and review the run

Reload the saved model through the repository API and require identical test predictions. Only after this passes does the manifest mark reload verification complete. Share the printed SHA, environment, selected input and checksum, shapes, class mapping, history, classification metrics, novelty flag counts, timings, reload result, and output folder for review.


In [ ]:
reload_start = perf_counter()
reloaded_model = TabularTransformerClassifier.load(checkpoint_path, device=device)
reloaded_predictions = reloaded_model.predict(dataset.test.features)
reload_seconds = perf_counter() - reload_start
assert reloaded_model.class_ids == model.class_ids
assert np.array_equal(reloaded_predictions, test_predictions), 'Checkpoint reload changed test predictions'
manifest['checkpoint_reload_verified'] = True
manifest['timings_seconds']['checkpoint_reload'] = reload_seconds
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Checkpoint reload predictions: identical')
print('Source SHA:', checked_out_sha)
print('Environment:', environment)
print('Input:', manifest['input'])
print('Split shapes:', manifest['split']['shapes'], 'class mapping:', dataset.class_names)
print('Training history:', history_dict)
print('Classification metrics:', metrics)
print('Novelty flag counts only:', novelty_counts)
print('Timings seconds:', manifest['timings_seconds'])
print('Manifest:', manifest_path)
